In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Solar_Footprints_V2.csv')

In [3]:
column_rename = {
    'Shape__Area': 'Polygon_Area',
    'Shape__Length': 'Polygon_Perimeter',
    'HIFLD ID (GTET 100 Max Voltage)': 'Substation_ID_GTET100',
    'Substation CASIO Name': 'Substation_Name_CAISO',
    'Solar Technoeconomic Intersection': 'Grid_Zone'
}
df = df.rename(columns=column_rename)

print("Columns after renaming:")
print(df.columns.tolist())

Columns after renaming:
['OBJECTID', 'County', 'Acres', 'Install Type', 'Urban or Rural', 'Combined Class', 'Distance to Substation (Miles) GTET 100 Max Voltage', 'Percentile (GTET 100 Max Voltage)', 'Substation Name GTET 100 Max Voltage', 'Substation_ID_GTET100', 'Distance to Substation (Miles) GTET 200 Max Voltage', 'Percentile (GTET 200 Max Voltage)', 'Substation Name GTET 200 Max Voltage', 'HIFLD ID (GTET 200 Max Voltage)', 'Distance to Substation (Miles) CAISO', 'Percentile (CAISO)', 'Substation_Name_CAISO', 'HIFLD ID (CAISO)', 'Grid_Zone', 'Polygon_Area', 'Polygon_Perimeter']


In [4]:
# Standardize categorical values
df['Install Type'] = df['Install Type'].str.title()
df['Urban or Rural'] = df['Urban or Rural'].str.title()
df['County'] = df['County'].str.title().str.replace(' County', '')

# Handle missing values
df['Grid_Zone'] = df['Grid_Zone'].fillna('Unknown')
df['Acres'] = pd.to_numeric(df['Acres'], errors='coerce')

print("\nMissing values after cleaning:")
print(df[['Grid_Zone', 'Acres']].isnull().sum())


Missing values after cleaning:
Grid_Zone    0
Acres        0
dtype: int64


In [5]:
# Land use categorization
df['Size_Category'] = pd.cut(
    df['Acres'],
    bins=[0, 1, 5, 10, np.inf],
    labels=['Small(<1)', 'Medium(1-5)', 'Large(5-10)', 'X-Large(10+)']
)

In [6]:
# Infrastructure scoring
percentile_cols = ['Percentile (GTET 100 Max Voltage)', 'Percentile (CAISO)']
for col in percentile_cols:
    df[col] = df[col].astype(str).str.extract(r'(\d+)').astype(float)
    df[col] = df[col].fillna(0)

df['Infrastructure_Score'] = df[percentile_cols].mean(axis=1).round(2)

In [7]:
# Grid accessibility metric
substation_cols = [col for col in df.columns if 'Distance to Substation' in col]
df['Avg_Distance_Substation'] = df[substation_cols].mean(axis=1).round(2)

# Carbon offset estimate (if annual production available)
if 'Estimated_Annual_Production_kWh' in df.columns:
    df['Carbon_Offset_tons'] = (df['Estimated_Annual_Production_kWh'] * 0.5 / 2000).round(2)
else:
    df['Carbon_Offset_tons'] = np.nan

print("\nSample of engineered features:")
print(df[['Acres', 'Size_Category', 'Infrastructure_Score', 'Avg_Distance_Substation', 'Carbon_Offset_tons']].head())


Sample of engineered features:
      Acres Size_Category  Infrastructure_Score  Avg_Distance_Substation  \
0  1.672639   Medium(1-5)                   0.0                     1.57   
1  1.897078   Medium(1-5)                   0.0                     1.67   
2  1.275783   Medium(1-5)                   0.0                     1.83   
3  1.424286   Medium(1-5)                   0.0                     1.91   
4  0.984429     Small(<1)                  12.5                     2.08   

   Carbon_Offset_tons  
0                 NaN  
1                 NaN  
2                 NaN  
3                 NaN  
4                 NaN  


In [8]:
# County-Level Sustainability Metrics
print(f"Total unique counties: {df['County'].nunique()}")
print("Counties:", df['County'].unique())

county_stats = df.groupby('County').agg(
    Total_Installations=('OBJECTID', 'count'),
    Avg_Acres=('Acres', 'mean'),
    Median_Acres=('Acres', 'median'),
    Rooftop_Pct=('Install Type', lambda x: (x == 'Rooftop').mean()),
    Ground_Pct=('Install Type', lambda x: (x == 'Ground').mean()),
    Infrastructure_Score=('Infrastructure_Score', 'mean'),
    Avg_Distance_Substation=('Avg_Distance_Substation', 'mean'),
    Carbon_Offset_tons=('Carbon_Offset_tons', 'sum')
).sort_values('Total_Installations', ascending=False).reset_index()

print("\nTop 5 Counties by Installations:")
print(county_stats.head())

Total unique counties: 51
Counties: ['Alameda' 'Amador' 'Butte' 'Calaveras' 'Colusa' 'Contra Costa'
 'El Dorado' 'Fresno' 'Glenn' 'Humboldt' 'Imperial' 'Inyo' 'Kern' 'Kings'
 'Lake' 'Lassen' 'Los Angeles' 'Madera' 'Marin' 'Mendocino' 'Merced'
 'Mono' 'Monterey' 'Napa' 'Nevada' 'Orange' 'Placer' 'Plumas' 'Riverside'
 'Sacramento' 'San Benito' 'San Bernardino' 'San Diego' 'San Francisco'
 'San Joaquin' 'San Luis Obispo' 'San Mateo' 'Santa Barbara' 'Santa Clara'
 'Santa Cruz' 'Shasta' 'Solano' 'Sonoma' 'Stanislaus' 'Sutter' 'Tehama'
 'Tulare' 'Tuolumne' 'Ventura' 'Yolo' 'Yuba']

Top 5 Counties by Installations:
           County  Total_Installations  Avg_Acres  Median_Acres  Rooftop_Pct  \
0     Los Angeles                  681  16.277480      2.450500     0.538913   
1            Kern                  523  59.502892      4.089526     0.065010   
2       San Diego                  391   3.047246      1.262166     0.531969   
3          Fresno                  372  22.749508      2.084150 

In [9]:
# Urban/Rural & Size Distribution
urban_rural_pivot = pd.pivot_table(
    df,
    index='Urban or Rural',
    columns='Size_Category',
    values='OBJECTID',
    aggfunc='count',
    fill_value=0
)

print("\nUrban/Rural Distribution by Size Category:")
print(urban_rural_pivot)


Urban/Rural Distribution by Size Category:
Size_Category   Small(<1)  Medium(1-5)  Large(5-10)  X-Large(10+)
Urban or Rural                                                   
Rural                 319          668          254           525
Urban                1142         1971          258           260


/var/folders/xf/c8cpvm517t9bg44l_slp0n6c0000gn/T/ipykernel_16923/2250378483.py:2: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  urban_rural_pivot = pd.pivot_table(


In [10]:
# Infrastructure Readiness Heatmap (County Level)
infra_heatmap = pd.crosstab(
    df['Percentile (GTET 100 Max Voltage)'],
    df['Percentile (CAISO)'],
    normalize='index'
).round(2)

print("\nInfrastructure Capacity Alignment Heatmap:")
print(infra_heatmap)


Infrastructure Capacity Alignment Heatmap:
Percentile (CAISO)                 0.0   25.0  50.0  75.0
Percentile (GTET 100 Max Voltage)                        
0.0                                0.31  0.22  0.26  0.21
25.0                               0.39  0.17  0.21  0.23
50.0                               0.14  0.42  0.25  0.18
75.0                               0.16  0.19  0.28  0.38


In [11]:
# County-Level Carbon Impact (if applicable)
if df['Carbon_Offset_tons'].notnull().any():
    county_carbon = df.groupby('County')['Carbon_Offset_tons'].sum().sort_values(ascending=False)
    print("\nTop 5 Counties by Estimated Annual CO2 Offset (tons):")
    print(county_carbon.head())
else:
    print("No annual production data available for carbon offset calculation.")

No annual production data available for carbon offset calculation.


In [12]:
df.to_csv('Cleaned_Solar_Data.csv', index=False)
county_stats.to_csv('County_Sustainability_Metrics.csv', index=False)
print("\nCleaned data and county metrics exported successfully!")


Cleaned data and county metrics exported successfully!
